In [1]:
!pip install selenium webdriver-manager


In [1]:
import time
import pandas as pd
from datetime import datetime, timezone

from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

c:\Users\chado\anaconda3\lib\site-packages\pandas\core\arrays\masked.py:60: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


In [2]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options

options = Options()
options.add_argument("--start-maximized")
options.add_argument("--disable-blink-features=AutomationControlled")

service = Service("C:/chromedriver/chromedriver-win64/chromedriver.exe")

driver = webdriver.Chrome(service=service, options=options)

driver.get("https://www.google.com")
print(driver.title)

NoSuchWindowException: Message: no such window: target window already closed
from unknown error: web view not found
  (Session info: chrome=144.0.7559.110)
Stacktrace:
Symbols not available. Dumping unresolved backtrace:
	0x7ff7e37088d5
	0x7ff7e3708930
	0x7ff7e34e165d
	0x7ff7e34b91b1
	0x7ff7e35697e6
	0x7ff7e358a5c2
	0x7ff7e352ac29
	0x7ff7e352ba93
	0x7ff7e3a20620
	0x7ff7e3a1af60
	0x7ff7e3a396c6
	0x7ff7e3725dd4
	0x7ff7e372ed7c
	0x7ff7e3711ff4
	0x7ff7e37121a5
	0x7ff7e36f7ed2
	0x7ffee159e8d7
	0x7ffee2a6c53c


In [4]:
# ======================================================
# Selenium setup
# ======================================================
options = Options()
options.add_argument("--start-maximized")
options.add_argument("--disable-blink-features=AutomationControlled")

service = Service("C:/chromedriver/chromedriver-win64/chromedriver.exe")
driver = webdriver.Chrome(service=service, options=options)


# ======================================================
# Helper: extract ALL spec key–value pairs (ROBUST)
# ======================================================
def extract_specs_dict(driver):
    specs = {}

    # each spec row contains a label + value
    rows = driver.find_elements(By.XPATH, "//div[.//span]")

    for row in rows:
        try:
            spans = row.find_elements(By.XPATH, ".//span")
            if len(spans) < 2:
                continue

            key = spans[0].text.strip().lower()
            value = spans[-1].text.strip()

            if key and value:
                specs[key] = value
        except:
            continue

    return specs


def extract_location(driver):
    try:
        container = driver.find_element(By.XPATH, "//*[@aria-label='Location']")
        # get the deepest visible text node
        text = container.text.strip()
        return text if text else None
    except:
        return None

# ======================================================
# Step 1 scraper for ONE listing page
# ======================================================
def scrape_step1_from_listing(driver, listing_url):
    driver.get(listing_url)

    WebDriverWait(driver, 20).until(
        EC.presence_of_element_located((By.TAG_NAME, "body"))
    )

    time.sleep(2)  # let React fully render

    specs = extract_specs_dict(driver)

    return {
        "listing_url": listing_url,

        # STEP 1 – vehicle identity
        "brand": specs.get("brand"),
        "model": specs.get("model"),
        "year": specs.get("year"),
        "fuel_type": specs.get("fuel type"),
        "transmission": specs.get("transmission type"),
        "body_type": specs.get("body type"),
        "condition": specs.get("condition"),
        "city": extract_location(driver),


        "scraped_at": datetime.now(timezone.utc).replace(microsecond=0)
    }


# ======================================================
# Step 0: paginate and collect listing URLs
# ======================================================
BASE_URL = "https://www.dubizzle.com.eg"
SEARCH_URL = "https://www.dubizzle.com.eg/en/vehicles/cars-for-sale/q-cars/"

listing_urls = set()
MAX_PAGES = 10  # increase later

for page in range(1, MAX_PAGES + 1):
    page_url = f"{SEARCH_URL}?page={page}"
    print(f"\nScraping search page {page}: {page_url}")

    driver.get(page_url)

    try:
        WebDriverWait(driver, 20).until(
            EC.presence_of_element_located((By.CSS_SELECTOR, "h2"))
        )
    except:
        print("No listings loaded — stopping pagination")
        break

    listings = driver.find_elements(By.CSS_SELECTOR, "article")
    print("Listings found:", len(listings))

    for listing in listings:
        try:
            link_el = listing.find_element(By.XPATH, ".//a[contains(@href, '/ad/')]")
            href = link_el.get_attribute("href")
            url = href if href.startswith("http") else BASE_URL + href
            listing_urls.add(url)
        except:
            pass

print(f"\nUnique listing URLs collected: {len(listing_urls)}")


# ======================================================
# Step 1: visit listing pages and extract vehicle identity
# ======================================================
step1_data = []

extraction_start_time = datetime.now(timezone.utc)
print("Step-1 extraction started at:", extraction_start_time)

# IMPORTANT: start small
for i, url in enumerate(list(listing_urls)[:len(listing_urls)], start=1):
    listing_start_time = datetime.now(timezone.utc)

    try:
        row = scrape_step1_from_listing(driver, url)
        step1_data.append(row)
    except Exception as e:
        print("Failed:", url, e)

    listing_end_time = datetime.now(timezone.utc)
    listing_duration = (listing_end_time - listing_start_time).total_seconds()

driver.quit()

extraction_end_time = datetime.now(timezone.utc)
extraction_duration = (extraction_end_time - extraction_start_time).total_seconds()

print("\nStep-1 extraction ended at:", extraction_end_time)
print("Total Step-1 extraction duration (minutes):", extraction_duration / 60)


# ======================================================
# Store into DataFrame
# ======================================================
df_step1 = pd.DataFrame(step1_data)

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)
df_step1["scraped_at"] = df_step1["scraped_at"].dt.strftime("%Y-%m-%d %H:%M:%S")
df_step1.to_csv('step1_df.csv')


print("\nSTEP-1 DATA:")
print(df_step1)
print("\nTotal Step-1 vehicles scraped:", len(df_step1))


Scraping search page 1: https://www.dubizzle.com.eg/en/vehicles/cars-for-sale/q-cars/?page=1
Listings found: 46

Scraping search page 2: https://www.dubizzle.com.eg/en/vehicles/cars-for-sale/q-cars/?page=2
No listings loaded — stopping pagination

Unique listing URLs collected: 46
Step-1 extraction started at: 2026-01-28 19:14:15.759973+00:00
Failed: https://www.dubizzle.com.eg/en/ad/%D9%85%D8%B1%D8%B3%D9%8A%D8%AF%D8%B3-%D8%A8%D9%86%D8%B2-gla-2018-ID206667336.html Message: no such window: target window already closed
from unknown error: web view not found
  (Session info: chrome=144.0.7559.97)
Stacktrace:
Symbols not available. Dumping unresolved backtrace:
	0x7ff79a5b88d5
	0x7ff79a5b8930
	0x7ff79a39165d
	0x7ff79a3691b1
	0x7ff79a4197e6
	0x7ff79a43a5c2
	0x7ff79a3dac29
	0x7ff79a3dba93
	0x7ff79a8d0620
	0x7ff79a8caf60
	0x7ff79a8e96c6
	0x7ff79a5d5dd4
	0x7ff79a5ded7c
	0x7ff79a5c1ff4
	0x7ff79a5c21a5
	0x7ff79a5a7ed2
	0x7ffee159e8d7
	0x7ffee2a6c53c

Failed: https://www.dubizzle.com.eg/en/ad/me

KeyError: 'scraped_at'

In [38]:
# ======================================================
# Selenium setup
# ======================================================
options = Options() 
options.add_argument("--start-maximized")
options.add_argument("--disable-blink-features=AutomationControlled")

service = Service("C:/chromedriver/chromedriver-win64/chromedriver.exe")
driver = webdriver.Chrome(service=service, options=options)


# ======================================================
# Helper: extract ALL spec key–value pairs (ROBUST)
# ======================================================
def extract_specs_dict(driver):
    specs = {}

    # each spec row contains a label + value
    rows = driver.find_elements(By.XPATH, "//div[.//span]")

    for row in rows:
        try:
            spans = row.find_elements(By.XPATH, ".//span")
            if len(spans) < 2:
                continue

            key = spans[0].text.strip().lower()
            value = spans[-1].text.strip()

            if key and value:
                specs[key] = value
        except:
            continue

    return specs


def extract_location(driver):
    try:
        container = driver.find_element(By.XPATH, "//*[@aria-label='Location']")
        # get the deepest visible text node
        text = container.text.strip()
        return text if text else None
    except:
        return None

# ======================================================
# Step 1 scraper for ONE listing page
# ======================================================
def scrape_step1_from_listing(driver, listing_url):
    driver.get(listing_url)

    WebDriverWait(driver, 20).until(
        EC.presence_of_element_located((By.TAG_NAME, "body"))
    )

    time.sleep(2)  # let React fully render

    specs = extract_specs_dict(driver)

    return {
        "listing_url": listing_url,

        # STEP 1 – vehicle identity
        "brand": specs.get("brand"),
        "model": specs.get("model"),
        "year": specs.get("year"),
        "fuel_type": specs.get("fuel type"),
        "transmission": specs.get("transmission type"),
        "body_type": specs.get("body type"),
        "engine_capacity": specs.get("engine capacity (cc)"), 

        "scraped_at": datetime.now(timezone.utc).replace(microsecond=0)
    }


# ======================================================
# Step 0: paginate and collect listing URLs
# ======================================================
BASE_URL = "https://www.dubizzle.com.eg"
SEARCH_URL = "https://www.dubizzle.com.eg/en/vehicles/cars-for-sale/q-cars/"

listing_urls = set()
MAX_PAGES = 199  # increase later

for page in range(186, MAX_PAGES + 1):
    page_url = f"{SEARCH_URL}?page={page}"
    print(f"\nScraping search page {page}: {page_url}")

    driver.get(page_url)

    try:
        WebDriverWait(driver, 20).until(
            EC.presence_of_element_located((By.CSS_SELECTOR, "h2"))
        )
    except:
        print("No listings loaded — stopping pagination")
        break

    listings = driver.find_elements(By.CSS_SELECTOR, "article")
    print("Listings found:", len(listings))

    for listing in listings:
        try:
            link_el = listing.find_element(By.XPATH, ".//a[contains(@href, '/ad/')]")
            href = link_el.get_attribute("href")
            url = href if href.startswith("http") else BASE_URL + href
            listing_urls.add(url)
        except:
            pass

print(f"\nUnique listing URLs collected: {len(listing_urls)}")


# ======================================================
# Step 1: visit listing pages and extract vehicle identity
# ======================================================
step1_data = []

extraction_start_time = datetime.now(timezone.utc)
print("Step-1 extraction started at:", extraction_start_time)

# IMPORTANT: start small
for i, url in enumerate(list(listing_urls)[:len(listing_urls)], start=1):
    listing_start_time = datetime.now(timezone.utc)

    try:
        row = scrape_step1_from_listing(driver, url)
        step1_data.append(row)
        print(f'Car {i} scraped successfully')
    except Exception as e:
        print("Failed:", url, e)

    listing_end_time = datetime.now(timezone.utc)
    listing_duration = (listing_end_time - listing_start_time).total_seconds()

driver.quit()

extraction_end_time = datetime.now(timezone.utc)
extraction_duration = (extraction_end_time - extraction_start_time).total_seconds()

print("\nStep-1 extraction ended at:", extraction_end_time)
print("Total Step-1 extraction duration (minutes):", extraction_duration / 60)


# ======================================================
# Store into DataFrame
# ======================================================
df_step1_2 = pd.DataFrame(step1_data)

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)
df_step1_2["scraped_at"] = df_step1_2["scraped_at"].dt.strftime("%Y-%m-%d %H:%M:%S")
df_step1_2.to_csv('step1_df_2.csv')


print("\nSTEP-1 DATA:")
print(df_step1_2)
print("\nTotal Step-1 vehicles scraped:", len(df_step1_2))


Scraping search page 186: https://www.dubizzle.com.eg/en/vehicles/cars-for-sale/q-cars/?page=186
Listings found: 46

Scraping search page 187: https://www.dubizzle.com.eg/en/vehicles/cars-for-sale/q-cars/?page=187
Listings found: 46

Scraping search page 188: https://www.dubizzle.com.eg/en/vehicles/cars-for-sale/q-cars/?page=188
Listings found: 46

Scraping search page 189: https://www.dubizzle.com.eg/en/vehicles/cars-for-sale/q-cars/?page=189
Listings found: 46

Scraping search page 190: https://www.dubizzle.com.eg/en/vehicles/cars-for-sale/q-cars/?page=190
Listings found: 46

Scraping search page 191: https://www.dubizzle.com.eg/en/vehicles/cars-for-sale/q-cars/?page=191
Listings found: 46

Scraping search page 192: https://www.dubizzle.com.eg/en/vehicles/cars-for-sale/q-cars/?page=192
Listings found: 46

Scraping search page 193: https://www.dubizzle.com.eg/en/vehicles/cars-for-sale/q-cars/?page=193
Listings found: 46

Scraping search page 194: https://www.dubizzle.com.eg/en/vehicl

In [ ]:
df_step1_2

,listing_url,brand,model,year,fuel_type,transmission,body_type,engine_capacity,scraped_at
0,https://www.dubizzle.com.eg/en/ad/%D8%A8%D9%89-%D9%88%D8%A7%D9%89-%D8%AF%D9%89-%D8%B3%D9%88%D9%86%D8%AC-%D8%A8%D9%84%D8%B3-byd-song-plus-605km-smart-ID206522694.html,BYD,Song Plus,2025,Electric,Automatic,SUV,None,2026-01-30 18:09:21
1,https://www.dubizzle.com.eg/en/ad/audi-a5-2019-ID206981170.html,Audi,A5,2019,Benzine,Automatic,Sedan,None,2026-01-30 18:09:41
2,https://www.dubizzle.com.eg/en/ad/%D8%B4%D9%8A%D8%B1%D9%89-%D8%AA%D9%8A%D8%AC%D9%88-8-2022-7-%D8%B1%D8%A7%D9%83%D8%A8-%D9%83%D9%88%D9%81%D9%88%D8%B1%D8%AA-ID205828795.html,Chery,Tiggo 8,2022,Benzine,Automatic,SUV,1500,2026-01-30 18:10:03
3,https://www.dubizzle.com.eg/en/ad/%D9%85%D9%8A%D8%AA%D8%B3%D9%88%D8%A8%D9%8A%D8%B4%D9%8A-%D9%84%D8%A7%D9%86%D8%B3%D8%B1-1997-ID206982043.html,Mitsubishi,Lancer,1997,Benzine,Automatic,Sedan,1300,2026-01-30 18:10:16
4,https://www.dubizzle.com.eg/en/ad/%D9%87%D9%8A%D9%88%D9%86%D8%AF%D8%A7%D9%8A-%D8%A5%D9%84%D9%8A%D9%86%D8%AA%D8%B1%D8%A7-2025-cn7-ID206879831.html,Hyundai,Elantra,2026,Benzine,Automatic,Sedan,1500,2026-01-30 18:10:33
...,...,...,...,...,...,...,...,...,...
563,https://www.dubizzle.com.eg/en/ad/bmw-x1-ID206988753.html,BMW,IX1,2020,Benzine,Automatic,SUV,1500,2026-01-30 19:47:45
564,https://www.dubizzle.com.eg/en/ad/chevrolet-aveo-2011-ID206984902.html,Chevrolet,Aveo,2011,Benzine,Manual,Sedan,1600,2026-01-30 19:47:53
565,https://www.dubizzle.com.eg/en/ad/proton-wira-2004-ID206983707.html,Proton,Wira,2004,Diesel,Manual,Sedan,1300,2026-01-30 19:47:58
566,https://www.dubizzle.com.eg/en/ad/%D8%A8%D9%8A%D8%AC%D9%88-3008-2025-peugeot-%D8%A7%D8%B3%D8%AA%D9%8A%D8%B1%D8%A7%D8%AF-%D8%AE%D9%84%D9%8A%D8%AC%D9%8A-%D8%A7%D8%B3%D8%AA%D9%84%D9%85-%D8%A8%D8%A3%D9%82%D9%84-%D9%85%D9%82%D8%AF%D9%85-%D9%88%D9%81%D9%88%D8%A7%D8%A6%D8%AF-%D8%AA%D8%A8%D8%AF%D8%A3-%D9%85%D9%8610-ID206975603.html,Peugeot,3008,2025,Benzine,Automatic,SUV,None,2026-01-30 19:48:05


In [4]:
df = pd.read_csv('step1_df.csv')

In [39]:
df1 = pd.concat([df, df_step1_2], ignore_index=True)

In [ ]:
df1

,Unnamed: 0.1,Unnamed: 0,listing_url,brand,model,year,fuel_type,transmission,body_type,engine_capacity,scraped_at
0,0,0.0,https://www.dubizzle.com.eg/en/ad/%D8%A8%D9%8A...,BMW,523,1999.0,Benzine,Automatic,Sedan,2500.0,2026-01-16 21:04:24
1,1,1.0,https://www.dubizzle.com.eg/en/ad/%D9%87%D9%8A...,Hyundai,Elantra,2026.0,Benzine,Automatic,Sedan,NaN,2026-01-16 21:04:33
2,2,2.0,https://www.dubizzle.com.eg/en/ad/%D9%81%D9%88...,Volkswagen,Golf,2010.0,Benzine,Automatic,Hatchback,1400.0,2026-01-16 21:04:45
3,3,3.0,https://www.dubizzle.com.eg/en/ad/citroen-ds4-...,Citroen,DS4,2023.0,Benzine,Automatic,Hatchback,1600.0,2026-01-16 21:04:59
4,4,4.0,https://www.dubizzle.com.eg/en/ad/%D9%87%D9%8A...,Hyundai,Elantra,2026.0,Benzine,Automatic,Sedan,1600.0,2026-01-16 21:05:08
...,...,...,...,...,...,...,...,...,...,...,...
6711,6711,NaN,https://www.dubizzle.com.eg/en/ad/bmw-x1-ID206...,BMW,IX1,2020.0,Benzine,Automatic,SUV,1500.0,2026-01-30 19:47:45
6712,6712,NaN,https://www.dubizzle.com.eg/en/ad/chevrolet-av...,Chevrolet,Aveo,2011.0,Benzine,Manual,Sedan,1600.0,2026-01-30 19:47:53
6713,6713,NaN,https://www.dubizzle.com.eg/en/ad/proton-wira-...,Proton,Wira,2004.0,Diesel,Manual,Sedan,1300.0,2026-01-30 19:47:58
6714,6714,NaN,https://www.dubizzle.com.eg/en/ad/%D8%A8%D9%8A...,Peugeot,3008,2025.0,Benzine,Automatic,SUV,NaN,2026-01-30 19:48:05


In [8]:
df1.drop(['Unnamed: 0.1',	'Unnamed: 0'], axis = 1)	
df1.to_csv('step1_df.csv')

NameError: name 'df1' is not defined

In [5]:
print(pd.unique(df1[['brand']].values.ravel()).size, pd.unique(df1[['model']].values.ravel()).size,pd.unique(df1[['brand', 'model']].values.ravel()).size)

NameError: name 'df1' is not defined

In [6]:
pd.unique(df1[['brand', 'model', 'year']].values.ravel()).size


NameError: name 'df1' is not defined

## Standardization

In [10]:
df = df.drop(['Unnamed: 0.1','Unnamed: 0'], axis = 1)	
df.to_csv('step1_df.csv')

In [14]:
df

,listing_url,brand,model,year,fuel_type,transmission,body_type,engine_capacity,scraped_at
0,https://www.dubizzle.com.eg/en/ad/%D8%A8%D9%8A...,BMW,523,1999.0,Benzine,Automatic,Sedan,2500.0,2026-01-16 21:04:24
1,https://www.dubizzle.com.eg/en/ad/%D9%87%D9%8A...,Hyundai,Elantra,2026.0,Benzine,Automatic,Sedan,NaN,2026-01-16 21:04:33
2,https://www.dubizzle.com.eg/en/ad/%D9%81%D9%88...,Volkswagen,Golf,2010.0,Benzine,Automatic,Hatchback,1400.0,2026-01-16 21:04:45
3,https://www.dubizzle.com.eg/en/ad/citroen-ds4-...,Citroen,DS4,2023.0,Benzine,Automatic,Hatchback,1600.0,2026-01-16 21:04:59
4,https://www.dubizzle.com.eg/en/ad/%D9%87%D9%8A...,Hyundai,Elantra,2026.0,Benzine,Automatic,Sedan,1600.0,2026-01-16 21:05:08
...,...,...,...,...,...,...,...,...,...
6711,https://www.dubizzle.com.eg/en/ad/bmw-x1-ID206...,BMW,IX1,2020.0,Benzine,Automatic,SUV,1500.0,2026-01-30 19:47:45
6712,https://www.dubizzle.com.eg/en/ad/chevrolet-av...,Chevrolet,Aveo,2011.0,Benzine,Manual,Sedan,1600.0,2026-01-30 19:47:53
6713,https://www.dubizzle.com.eg/en/ad/proton-wira-...,Proton,Wira,2004.0,Diesel,Manual,Sedan,1300.0,2026-01-30 19:47:58
6714,https://www.dubizzle.com.eg/en/ad/%D8%A8%D9%8A...,Peugeot,3008,2025.0,Benzine,Automatic,SUV,NaN,2026-01-30 19:48:05


In [12]:
df['listing_url'].duplicated().sum()

1133

In [15]:
df_duplicates = df[df['listing_url'].duplicated(keep=False)]


In [17]:
df_duplicates.sort_values(by = 'listing_url')

,listing_url,brand,model,year,fuel_type,transmission,body_type,engine_capacity,scraped_at
5347,https://www.dubizzle.com.eg/en/ad/%D8%A3%D8%B9...,Hyundai,Elantra,2025.0,Electric,Automatic,Sedan,NaN,2026-01-29 23:08:18
6244,https://www.dubizzle.com.eg/en/ad/%D8%A3%D8%B9...,Hyundai,Elantra,2025.0,Electric,Automatic,Sedan,NaN,2026-01-30 18:35:25
6157,https://www.dubizzle.com.eg/en/ad/%D8%A3%D8%B9...,Hyundai,Elantra,2025.0,Electric,Automatic,Sedan,NaN,2026-01-30 18:11:42
4161,https://www.dubizzle.com.eg/en/ad/%D8%A3%D8%B9...,Hyundai,Elantra,2025.0,Electric,Automatic,Sedan,NaN,2026-01-29 19:09:56
2293,https://www.dubizzle.com.eg/en/ad/%D8%A3%D9%88...,Opel,Astra,2020.0,Benzine,Automatic,Sedan,NaN,2026-01-17 05:53:33
...,...,...,...,...,...,...,...,...,...
3321,https://www.dubizzle.com.eg/en/ad/xpeng-g9-202...,XPeng,G9,2025.0,Electric,Automatic,SUV,NaN,2026-01-17 14:43:36
3389,https://www.dubizzle.com.eg/en/ad/zeekr-7x-202...,Zeekr,7X,2025.0,Electric,Automatic,SUV,NaN,2026-01-17 15:03:11
2975,https://www.dubizzle.com.eg/en/ad/zeekr-7x-202...,Zeekr,7X,2025.0,Electric,Automatic,SUV,NaN,2026-01-17 13:11:57
2339,https://www.dubizzle.com.eg/en/ad/zeekr-7x-202...,Zeekr,7X,2025.0,Electric,Automatic,SUV,310.0,2026-01-17 10:50:09


## Dropping duplicated urls

In [18]:
df_unique = df.drop_duplicates(subset=['listing_url'])
df_unique

,listing_url,brand,model,year,fuel_type,transmission,body_type,engine_capacity,scraped_at
0,https://www.dubizzle.com.eg/en/ad/%D8%A8%D9%8A...,BMW,523,1999.0,Benzine,Automatic,Sedan,2500.0,2026-01-16 21:04:24
1,https://www.dubizzle.com.eg/en/ad/%D9%87%D9%8A...,Hyundai,Elantra,2026.0,Benzine,Automatic,Sedan,NaN,2026-01-16 21:04:33
2,https://www.dubizzle.com.eg/en/ad/%D9%81%D9%88...,Volkswagen,Golf,2010.0,Benzine,Automatic,Hatchback,1400.0,2026-01-16 21:04:45
3,https://www.dubizzle.com.eg/en/ad/citroen-ds4-...,Citroen,DS4,2023.0,Benzine,Automatic,Hatchback,1600.0,2026-01-16 21:04:59
4,https://www.dubizzle.com.eg/en/ad/%D9%87%D9%8A...,Hyundai,Elantra,2026.0,Benzine,Automatic,Sedan,1600.0,2026-01-16 21:05:08
...,...,...,...,...,...,...,...,...,...
6709,https://www.dubizzle.com.eg/en/ad/volkswagen-t...,Volkswagen,Tayron,2026.0,Benzine,Automatic,SUV,1400.0,2026-01-30 19:47:27
6710,https://www.dubizzle.com.eg/en/ad/chevrolet-op...,Chevrolet,Optra,2007.0,Benzine,Automatic,Sedan,1600.0,2026-01-30 19:47:35
6711,https://www.dubizzle.com.eg/en/ad/bmw-x1-ID206...,BMW,IX1,2020.0,Benzine,Automatic,SUV,1500.0,2026-01-30 19:47:45
6712,https://www.dubizzle.com.eg/en/ad/chevrolet-av...,Chevrolet,Aveo,2011.0,Benzine,Manual,Sedan,1600.0,2026-01-30 19:47:53


In [29]:
df_unique.to_csv('unique_urls_df.csv')

In [24]:
df_duplicates = df_unique[df_unique.duplicated(subset=['brand', 'model', 'year', 'fuel_type', 'transmission', 'body_type', 'engine_capacity'], keep=False)]
df_duplicates


,listing_url,brand,model,year,fuel_type,transmission,body_type,engine_capacity,scraped_at
0,https://www.dubizzle.com.eg/en/ad/%D8%A8%D9%8A...,BMW,523,1999.0,Benzine,Automatic,Sedan,2500.0,2026-01-16 21:04:24
1,https://www.dubizzle.com.eg/en/ad/%D9%87%D9%8A...,Hyundai,Elantra,2026.0,Benzine,Automatic,Sedan,NaN,2026-01-16 21:04:33
3,https://www.dubizzle.com.eg/en/ad/citroen-ds4-...,Citroen,DS4,2023.0,Benzine,Automatic,Hatchback,1600.0,2026-01-16 21:04:59
4,https://www.dubizzle.com.eg/en/ad/%D9%87%D9%8A...,Hyundai,Elantra,2026.0,Benzine,Automatic,Sedan,1600.0,2026-01-16 21:05:08
5,https://www.dubizzle.com.eg/en/ad/renault-flue...,Renault,Fluence,2014.0,Benzine,Automatic,Sedan,1600.0,2026-01-16 21:05:16
...,...,...,...,...,...,...,...,...,...
6706,https://www.dubizzle.com.eg/en/ad/%D8%B4%D9%8A...,Chevrolet,Optra,2026.0,Benzine,Automatic,Sedan,1500.0,2026-01-30 19:47:01
6708,https://www.dubizzle.com.eg/en/ad/mercedes-ben...,Mercedes-Benz,EQA 260,2025.0,Electric,Automatic,SUV,NaN,2026-01-30 19:47:20
6709,https://www.dubizzle.com.eg/en/ad/volkswagen-t...,Volkswagen,Tayron,2026.0,Benzine,Automatic,SUV,1400.0,2026-01-30 19:47:27
6710,https://www.dubizzle.com.eg/en/ad/chevrolet-op...,Chevrolet,Optra,2007.0,Benzine,Automatic,Sedan,1600.0,2026-01-30 19:47:35


In [36]:
counts_df = df_unique.groupby(['brand', 'model', 'year', 'fuel_type', 'transmission', 'body_type', 'engine_capacity']).size().reset_index(name='count').sort_values(by = 'count', ascending=False)
counts_df
counts_df.to_csv('counts.csv')

## Dropping duplicated cars

In [34]:
cols = [
    'brand',
    'model',
    'year',
    'fuel_type',
    'transmission',
    'body_type',
    'engine_capacity'
]

df_unique_cars = df_unique.drop_duplicates(subset=cols, keep='first')
df_unique_cars

,listing_url,brand,model,year,fuel_type,transmission,body_type,engine_capacity,scraped_at
0,https://www.dubizzle.com.eg/en/ad/%D8%A8%D9%8A...,BMW,523,1999.0,Benzine,Automatic,Sedan,2500.0,2026-01-16 21:04:24
1,https://www.dubizzle.com.eg/en/ad/%D9%87%D9%8A...,Hyundai,Elantra,2026.0,Benzine,Automatic,Sedan,NaN,2026-01-16 21:04:33
2,https://www.dubizzle.com.eg/en/ad/%D9%81%D9%88...,Volkswagen,Golf,2010.0,Benzine,Automatic,Hatchback,1400.0,2026-01-16 21:04:45
3,https://www.dubizzle.com.eg/en/ad/citroen-ds4-...,Citroen,DS4,2023.0,Benzine,Automatic,Hatchback,1600.0,2026-01-16 21:04:59
4,https://www.dubizzle.com.eg/en/ad/%D9%87%D9%8A...,Hyundai,Elantra,2026.0,Benzine,Automatic,Sedan,1600.0,2026-01-16 21:05:08
...,...,...,...,...,...,...,...,...,...
6690,https://www.dubizzle.com.eg/en/ad/%D8%AC%D9%8A...,Geely,Ck2,2009.0,Benzine,Manual,Sedan,1500.0,2026-01-30 19:44:59
6701,https://www.dubizzle.com.eg/en/ad/%D9%81%D9%88...,Volkswagen,Tiguan,2026.0,Benzine,Automatic,Hatchback,1400.0,2026-01-30 19:46:21
6704,https://www.dubizzle.com.eg/en/ad/%D8%B4%D9%8A...,Chevrolet,Cruze,2009.0,Benzine,Automatic,Sedan,1600.0,2026-01-30 19:46:46
6711,https://www.dubizzle.com.eg/en/ad/bmw-x1-ID206...,BMW,IX1,2020.0,Benzine,Automatic,SUV,1500.0,2026-01-30 19:47:45


In [38]:
df_unique_cars.groupby(['brand', 'model', 'year', 'fuel_type', 'transmission', 'body_type', 'engine_capacity']).size().reset_index(name='count').sort_values(by = 'count', ascending=False)


,brand,model,year,fuel_type,transmission,body_type,engine_capacity,count
0,Alfa Romeo,156,2002.0,Benzine,Manual,Sedan,2000.0,1
1795,Mitsubishi,Pajero,2017.0,Benzine,Automatic,4X4,3500.0,1
1787,Mitsubishi,Mirage,2025.0,Benzine,Automatic,Hatchback,1200.0,1
1788,Mitsubishi,Outlander,2025.0,Benzine,Automatic,SUV,1600.0,1
1789,Mitsubishi,Pajero,1997.0,Benzine,Automatic,4X4,3000.0,1
...,...,...,...,...,...,...,...,...
897,Honda,Civic,1900.0,Benzine,Automatic,Sedan,1500.0,1
898,Honda,Civic,1979.0,Benzine,Automatic,Hatchback,1500.0,1
899,Honda,Civic,1989.0,Benzine,Manual,Sedan,1500.0,1
900,Honda,Civic,1989.0,Diesel,Manual,Sedan,1600.0,1


In [39]:
null_counts = (
    df_unique_cars.isna()
      .sum()
      .reset_index(name='null_count')
      .rename(columns={'index': 'column'})
)

In [40]:
null_counts

,column,null_count
0,listing_url,0
1,brand,8
2,model,8
3,year,8
4,fuel_type,1
5,transmission,1
6,body_type,5
7,engine_capacity,1103
8,scraped_at,0
